# Brazil Ecoregions and Copper Mines Analysis

This notebook visualizes the ecoregions of Brazil and highlights those containing copper mines.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 10)

## 1. Load Data

In [ ]:
# Load Brazil boundary from specified shapefile
admin_units_path = "data/ne_10m_admin_0_map_units/ne_10m_admin_0_map_units.shp"
admin_units = gpd.read_file(admin_units_path)

# Filter for Brazil
if 'ADMIN' in admin_units.columns:
    brazil = admin_units[admin_units['ADMIN'] == 'Brazil']
elif 'NAME' in admin_units.columns:
    brazil = admin_units[admin_units['NAME'] == 'Brazil']

# Load Ecoregions
ecoregions_path = "data/ecoregions/official/wwf_terr_ecos.shp"
ecoregions = gpd.read_file(ecoregions_path)

# Load Mining Facilities
facilities_path = "data/mining/jasansky/data/facilities.gpkg"
facilities = gpd.read_file(facilities_path)

print("Data loaded successfully.")

## 2. Process Ecoregions for Brazil

In [ ]:
# Ensure CRS match
if ecoregions.crs != brazil.crs:
    ecoregions = ecoregions.to_crs(brazil.crs)

# Clip Ecoregions to Brazil
# Using overlay intersection handles the boundaries better than clip sometimes for complex geometries
brazil_ecoregions = gpd.overlay(ecoregions, brazil, how='intersection')

# Calculate Area
# Project to an equal-area projection for accurate area calculation (e.g., SIRGAS 2000 / Brazil Polyconic or Albers)
# EPSG:5880 is Polyconic, often used for Brazil, or 102033 (South America Albers Equal Area)
# Simple approach: use South America Albers Equal Area
brazil_ecoregions_proj = brazil_ecoregions.to_crs("ESRI:102033")
brazil_ecoregions['area_km2'] = brazil_ecoregions_proj.geometry.area / 10**6

print(f"Number of ecoregions in Brazil: {len(brazil_ecoregions)}")
display(brazil_ecoregions[['ECO_NAME', 'area_km2']].head())

## 3. Plot 1: Brazil Ecoregions with Area

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))

# Plot all ecoregions colored by area
brazil_ecoregions.plot(column='area_km2', ax=ax, legend=True,
                       legend_kwds={'label': "Area (km²)", 'orientation': "horizontal"},
                       cmap='viridis', edgecolor='black', linewidth=0.3)

# Plot Brazil boundary on top for clarity
brazil.boundary.plot(ax=ax, color='black', linewidth=1)

ax.set_title("Ecoregions of Brazil by Area", fontsize=20)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4. Identify Ecoregions with Copper Mines

In [ ]:
# Filter for Copper mines
# Check columns first to be sure 'primary_commodity' or similar exists. Based on prev inspection it was 'primary_commodity'
copper_mines = facilities[facilities['primary_commodity'].str.contains('Copper', case=False, na=False)]

# Ensure CRS match
if copper_mines.crs != brazil.crs:
    copper_mines = copper_mines.to_crs(brazil.crs)

# Filter for mines within Brazil
# Spatial join or clip. Since points are simple, clip/intersection is fast.
brazil_copper_mines = gpd.clip(copper_mines, brazil)

print(f"Number of Copper mines in Brazil: {len(brazil_copper_mines)}")

# Identify ecoregions containing these mines
# Spatial join between Brazil Ecoregions and Brazil Copper Mines
mines_with_ecoregions = gpd.sjoin(brazil_copper_mines, brazil_ecoregions, how='inner', predicate='within')

# Get unique Eco IDs or Names that have mines
target_eco_names = mines_with_ecoregions['ECO_NAME'].unique()
target_ecoregions = brazil_ecoregions[brazil_ecoregions['ECO_NAME'].isin(target_eco_names)]

print(f"Number of ecoregions with Copper mines: {len(target_ecoregions)}")

## 5. Plot 2: Ecoregions containing Copper Mines

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))

# Plot Brazil background (faint)
brazil.plot(ax=ax, color='lightgrey', edgecolor='white')

# Plot Target Ecoregions
target_ecoregions.plot(ax=ax, color='lightblue', edgecolor='black', alpha=0.6, linewidth=0.5, label='Ecoregions with Copper Mines')

# Plot Mines
brazil_copper_mines.plot(ax=ax, color='red', markersize=20, label='Copper Mines', edgecolor='white', linewidth=0.5)

ax.set_title("Brazil Ecoregions Containing Copper Mines", fontsize=20)
ax.set_axis_off()

# Custom legend
import matplotlib.patches as mpatches
eco_patch = mpatches.Patch(color='lightblue', label='Ecoregions with Copper Mines')
mine_marker = plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Copper Mines')
plt.legend(handles=[eco_patch, mine_marker], loc='lower right', fontsize=12)

plt.tight_layout()
plt.show()